# ================================================================
# BOOTCAMP: Fundamentos de Ingeniería de Datos
# Semana 2: Práctica de CTEs y Window Functions
# ================================================================

**Instructor:** Luciano Argolo  
**Web:** lucianoargolo.com

---

## 🎯 Objetivos

1. Dominar **CTEs (Common Table Expressions)** para organizar queries complejas
2. Aprender **Window Functions** para análisis avanzados
3. Combinar ambas técnicas para resolver problemas reales
4. Practicar con datos reales de NYC Taxi Trips

---

## 📊 Dataset

Usaremos la tabla `samples.nyctaxi.trips` que contiene datos de viajes de taxis en Nueva York.

**Columnas principales:**
- `tpep_pickup_datetime`: Fecha y hora de inicio del viaje
- `tpep_dropoff_datetime`: Fecha y hora de fin del viaje
- `trip_distance`: Distancia del viaje en millas
- `fare_amount`: Tarifa del viaje
- `pickup_zip`: Código postal de inicio
- `dropoff_zip`: Código postal de destino

## Ejercicio 0.1: CTE básica - Estadísticas por zona

**Objetivo:** Usar una CTE para organizar cálculos.

**Pregunta:** Calcula el promedio de tarifa y distancia por zona de inicio (`pickup_zip`). Usa una CTE para calcular primero las estadísticas por zona, y luego muestra solo las zonas con más de 100 viajes.

In [0]:
%sql
-- Ejercicio 0.1: CTE básica - Estadísticas por zona
-- Usamos WITH para crear una CTE que calcula estadísticas por zona
-- Luego filtramos solo las zonas con más de 100 viajes

WITH estadisticas_por_zona AS (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
  GROUP BY pickup_zip
  HAVING cantidad_viajes > 100
)
SELECT 
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  distancia_promedio
FROM estadisticas_por_zona
ORDER BY cantidad_viajes DESC
LIMIT 20;

**Mismo resultado con subquery:** Más difícil de leer y no podés reutilizar la subconsulta. Con CTEs, si necesitás `estadisticas_por_zona` en otro lado, ya la tenés definida.

In [0]:
%sql
-- Ejercicio 0.1 (versión con subquery)

SELECT 
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  distancia_promedio
FROM (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
  GROUP BY pickup_zip
  HAVING cantidad_viajes > 100
) estadisticas_por_zona
ORDER BY cantidad_viajes DESC
LIMIT 20;

## Ejercicio 0.2: CTE múltiple - Análisis comparativo

**Objetivo:** Usar múltiples CTEs para organizar cálculos complejos.

**Pregunta:** Compara el promedio de tarifa por hora del día con el promedio general. Usa dos CTEs: una para calcular el promedio por hora, otra para calcular el promedio general, y luego únelas para mostrar la diferencia.

In [0]:
%sql
-- Ejercicio 0.2: CTE múltiple - Análisis comparativo
-- Usamos múltiples CTEs para organizar el cálculo
-- Primero calculamos promedio por hora, luego el promedio general

WITH promedio_por_hora AS (
  SELECT 
    HOUR(tpep_pickup_datetime) AS hora_del_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS (
  SELECT 
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
)
SELECT 
  p.hora_del_dia,
  p.cantidad_viajes,
  p.tarifa_promedio_hora,
  g.tarifa_promedio_total,
  ROUND(p.tarifa_promedio_hora - g.tarifa_promedio_total, 2) AS diferencia,
  ROUND((p.tarifa_promedio_hora - g.tarifa_promedio_total) * 100.0 / g.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM promedio_por_hora p
CROSS JOIN promedio_general g
ORDER BY p.hora_del_dia;

**Mismo resultado con subquery:** Acá se nota más la diferencia. Con CTEs cada paso tiene nombre y se lee de arriba para abajo. Con subqueries tenés que leer de adentro para afuera.

In [0]:
%sql
-- Ejercicio 0.2 (versión con subqueries)
-- Misma lógica que el CTE, pero anidando las queries
-- Más difícil de leer y debuguear — por eso preferimos CTEs

SELECT 
  p.hora_del_dia,
  p.cantidad_viajes,
  p.tarifa_promedio_hora,
  g.tarifa_promedio_total,
  ROUND(p.tarifa_promedio_hora - g.tarifa_promedio_total, 2) AS diferencia,
  ROUND((p.tarifa_promedio_hora - g.tarifa_promedio_total) * 100.0 / g.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM (
  SELECT 
    HOUR(tpep_pickup_datetime) AS hora_del_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
) p
CROSS JOIN (
  SELECT ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
) g
ORDER BY p.hora_del_dia;

# Un ejemplo de Usar Window functions para obtener resultados similares

In [0]:
%sql
SELECT 
  Distinct(HOUR(tpep_pickup_datetime)) AS hora_del_dia,
  ROUND(AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)), 2) AS tarifa_promedio_hora,
  ROUND(AVG(fare_amount) OVER (), 2) AS tarifa_promedio_total,
  ROUND(AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)) - AVG(fare_amount) OVER (), 2) AS diferencia,
  ROUND((AVG(fare_amount) OVER (PARTITION BY HOUR(tpep_pickup_datetime)) - AVG(fare_amount) OVER ()) * 100.0 / AVG(fare_amount) OVER (), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0;

## Ejercicio 0.3: CTE para limpieza de datos

**Objetivo:** Usar CTE para filtrar datos antes de analizar.

**Pregunta:** Calcula estadísticas de distancia solo para viajes válidos (distancia > 0, tarifa > 0). Usa una CTE para filtrar primero los datos válidos, y luego calcula mínimo, máximo, promedio y mediana.

In [0]:
%sql
-- Ejercicio 0.3: CTE para limpieza de datos
-- La CTE nos permite filtrar datos inválidos primero
-- Luego trabajamos solo con datos limpios

WITH viajes_validos AS (
  SELECT 
    trip_distance,
    fare_amount
  FROM samples.nyctaxi.trips
  WHERE trip_distance > 0
    AND fare_amount > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
)
SELECT 
  COUNT(*) AS total_viajes_validos,
  ROUND(MIN(trip_distance), 2) AS distancia_minima,
  ROUND(MAX(trip_distance), 2) AS distancia_maxima,
  ROUND(AVG(trip_distance), 2) AS distancia_promedio,
  ROUND(PERCENTILE(trip_distance, 0.5), 2) AS distancia_mediana
FROM viajes_validos;

## Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking

**Objetivo:** Usar ROW_NUMBER() para crear rankings.

**Pregunta:** Crea un ranking de los viajes más caros. Muestra los top 20 viajes con su fecha, distancia, tarifa y el ranking (1 = más caro).

In [0]:
%sql
-- Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking
-- ROW_NUMBER() asigna números únicos consecutivos (1, 2, 3...)
-- No hay empates, cada fila tiene un número único

SELECT 
  ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS ranking,
  tpep_pickup_datetime AS fecha,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20;

## Ejercicio 0.5: Window Function - RANK() y DENSE_RANK()

**Objetivo:** Diferenciar entre RANK(), DENSE_RANK() y ROW_NUMBER().

**Pregunta:** Para las 10 zonas con más viajes, crea tres rankings diferentes usando ROW_NUMBER(), RANK() y DENSE_RANK() ordenados por cantidad de viajes. ¿Notas la diferencia?

In [0]:
%sql
-- Ejercicio 0.5: RANK() y DENSE_RANK() con datos reales de NYC Taxi
-- Primero calculamos cantidad de viajes por zona, luego aplicamos las 3 funciones

WITH viajes_por_zona AS (
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
  GROUP BY pickup_zip
  ORDER BY cantidad_viajes DESC
  LIMIT 10
)
SELECT 
  pickup_zip,
  cantidad_viajes,
  ROW_NUMBER() OVER (ORDER BY cantidad_viajes DESC) AS ranking_row_number,
  RANK()       OVER (ORDER BY cantidad_viajes DESC) AS ranking_rank,
  DENSE_RANK() OVER (ORDER BY cantidad_viajes DESC) AS ranking_dense_rank
FROM viajes_por_zona
ORDER BY cantidad_viajes DESC;

## Ejercicio 0.6: Window Function - Comparar con promedio general

**Objetivo:** Usar AVG() OVER() para comparar con el promedio.

**Pregunta:** Para cada viaje, muestra la tarifa, la distancia, y cómo se compara la tarifa con el promedio general (diferencia y porcentaje de diferencia).

In [0]:
%sql
-- Ejercicio 0.6: Window Function - Comparar con promedio general
-- AVG() OVER() calcula el promedio de TODAS las filas sin agrupar
-- Esto nos permite comparar cada fila individual con el promedio general

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(AVG(fare_amount) OVER(), 2) AS tarifa_promedio_general,
  ROUND(fare_amount - AVG(fare_amount) OVER(), 2) AS diferencia_con_promedio,
  ROUND((fare_amount - AVG(fare_amount) OVER()) * 100.0 / AVG(fare_amount) OVER(), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fecha DESC
LIMIT 20;

## Ejercicio 0.7: Window Function - Promedio por partición (PARTITION BY)

**Objetivo:** Usar PARTITION BY para calcular promedios por grupo.

**Pregunta:** Para cada viaje, muestra: fecha, tarifa, zona de inicio (pickup_zip), y el promedio de tarifa para esa misma zona (ej: si es zona 10001, muestra el promedio de todos los viajes de la zona 10001).

**Diferencia clave con 0.6:** En 0.6 usamos `AVG() OVER()` sin PARTITION BY → promedio **general**. Acá usamos `PARTITION BY pickup_zip` → promedio **por zona**.

In [0]:
%sql
-- Ejercicio 0.7: PARTITION BY para calcular promedios por zona
-- AVG() OVER (PARTITION BY pickup_zip) calcula el promedio
-- solo dentro del grupo de viajes de la misma zona

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  pickup_zip,
  ROUND(AVG(fare_amount) OVER (PARTITION BY pickup_zip), 2) AS tarifa_promedio_zona,
  ROUND(fare_amount - AVG(fare_amount) OVER (PARTITION BY pickup_zip), 2) AS diferencia_con_zona
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
  AND pickup_zip IS NOT NULL
ORDER BY pickup_zip, fare_amount DESC
LIMIT 50;

## Ejercicio 0.8: Window Function - LAG() para comparar con anterior

**Objetivo:** Usar LAG() para comparar con el registro anterior.

**Pregunta:** Ordena los viajes por fecha y muestra: fecha, tarifa, y la tarifa del viaje anterior (si existe). Esto te permite ver si hay tendencias temporales.

In [0]:
%sql
-- Ejercicio 0.8: Window Function - LAG() para comparar con anterior
-- LAG() obtiene el valor de la fila anterior según el orden especificado
-- Útil para comparar valores consecutivos y detectar tendencias

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa_actual,
  LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime) AS tarifa_anterior,
  ROUND(fare_amount - LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime), 2) AS diferencia_con_anterior
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY tpep_pickup_datetime
LIMIT 50;

## Ejercicio 0.9: Window Function - SUM() acumulado

**Objetivo:** Usar SUM() OVER() para calcular acumulados.

**Pregunta:** Ordena los viajes por fecha y muestra: fecha, tarifa, y el total acumulado de tarifas hasta esa fecha (running total).

In [0]:
%sql
-- Ejercicio 0.9: Window Function - SUM() acumulado
-- SUM() OVER() con ORDER BY calcula un total acumulado (running total)
-- ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW suma desde el inicio hasta la fila actual

SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(SUM(fare_amount) OVER (
    ORDER BY tpep_pickup_datetime 
  ), 2) AS total_acumulado
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY tpep_pickup_datetime
LIMIT 100;

## Ejercicio 0.10: Combinando CTEs y Window Functions

**Objetivo:** Combinar ambas técnicas en una query compleja.

**Pregunta:** Usa CTEs para:
1. Filtrar viajes válidos (distancia > 0, tarifa > 0)
2. Calcular estadísticas por zona (promedio, máximo, mínimo)
3. Luego usa Window Functions para rankear las zonas por promedio de tarifa
4. Muestra el top 10 zonas más caras con su ranking

In [0]:
%sql
-- Ejercicio 0.10: Combinando CTEs y Window Functions
-- Este ejercicio demuestra cómo combinar ambas técnicas
-- CTEs para organizar y limpiar datos, Window Functions para análisis avanzado

WITH viajes_validos AS (
  -- Paso 1: Filtrar viajes válidos
  SELECT 
    pickup_zip,
    fare_amount,
    trip_distance
  FROM samples.nyctaxi.trips
  WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
),
estadisticas_por_zona AS (
  -- Paso 2: Calcular estadísticas por zona
  SELECT 
    pickup_zip,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
    ROUND(MAX(fare_amount), 2) AS tarifa_maxima,
    ROUND(MIN(fare_amount), 2) AS tarifa_minima
  FROM viajes_validos
  GROUP BY pickup_zip
  HAVING COUNT(*) >= 50  -- Solo zonas con al menos 50 viajes
)
SELECT 
  -- Paso 3: Usar Window Function para rankear
  ROW_NUMBER() OVER (ORDER BY tarifa_promedio DESC) AS ranking,
  pickup_zip,
  cantidad_viajes,
  tarifa_promedio,
  tarifa_maxima,
  tarifa_minima
FROM estadisticas_por_zona
ORDER BY tarifa_promedio DESC
LIMIT 10;

---

## 💡 Resumen de Conceptos Clave

### CTEs (Common Table Expressions)
- **Cuándo usarlas:** Para organizar queries complejas, reutilizar subconsultas, mejorar legibilidad
- **Sintaxis:** `WITH nombre_cte AS (SELECT ...)`
- **Ventajas:** Código más limpio, fácil de mantener, permite múltiples CTEs en una query

### Window Functions
- **ROW_NUMBER():** Asigna números únicos consecutivos (1, 2, 3...)
- **RANK():** Ranking con saltos en empates (1, 2, 2, 4...)
- **DENSE_RANK():** Ranking sin saltos en empates (1, 2, 2, 3...)
- **AVG() OVER():** Calcula promedio sin agrupar filas
- **SUM() OVER():** Calcula suma acumulada
- **LAG():** Obtiene valor de la fila anterior
- **PARTITION BY:** Agrupa filas antes de aplicar la función

### Combinando CTEs y Window Functions
- Usa CTEs para limpiar y preparar datos
- Usa Window Functions para análisis comparativos y rankings
- Ambas técnicas se complementan perfectamente

---

**¡Ahora estás listo para aplicar estas técnicas en el EDA de propiedades! 🚀**